# BM-GHSL Surface Match

## Methods

To understand how well NTL products capture built environments across contexts ranging from less to more developed countries, we assess regional differences in NTL products from two complementary perspectives. First, many studies rely on thresholding to derive binary representations both for NTL radiance and for the corresponding reference data used to evaluate model performance, such as population or infrastructure layers. NTL radiance and the corresponding comparison layers represent different quantities and units, and thus making them directly comparable first requires binarizing each layer. In this context, thresholds for radiance measures are relatively well established in the literature, whereas several studies have shown that choosing thresholds for binarizing comparison layers is highly sensitive to regional context.

To explore this issue, we use data from the Global Human Settlement Layer (GHSL) project, specifically the GHSL built-up surface product (GHS‑BUILT‑S R2023A). The dataset provides global maps of built-up surfaces at 100 m resolution, where each pixel value represents the proportion of built-up area within 100 square meters. In this project, we use 2020 layer, as it is the most recent epoch directly observed from satellite data rather than extrapolated. 

A threshold of 1\% (i.e., at least one percent of the pixel area is built-up) is commonly adopted to delineate urban areas, whereas recommended thresholds for rural areas are substantially higher, typically ranging from 20% to 50%. In our analysis, we apply two thresholds and compare the resulting patterns across products. The GHSL built-up data are first resampled to 500 m resolution to match the NTL composites, after which both the NTL layer and the GHSL built-up surface are binarized – classifying pixels as either lit or unlit, and as built-up or non–built–up – using thresholds of 1% and 20%. Each binarized nighttime light layer is then paired with each of the two GHSL layers in turn, and the resulting pairs are used to construct confusion matrices for subsequent analysis.

Given the pronounced differences in development levels both across and within African and European countries, we then use the Human Development Index (HDI) – a composite measure of national development that combines indicators of health, education, and income – to examine how national development relates to the accuracy metrics, namely F1 score, precision, and recall, that quantify the agreement between built-up and lit areas.

In [ ]:
import pandas as pd
import pycountry_convert as pc

from bokeh.plotting import output_notebook

from conflict_monitoring_ntl.viz import plot_scatter_bokeh

output_notebook()

Loading BokehJS ...

In [2]:
def country_to_continent(country_name):
    try:
        country_code = pc.country_name_to_country_alpha2(country_name)
        continent_code = pc.country_alpha2_to_continent_code(country_code)
        continent_name = pc.convert_continent_code_to_continent_name(continent_code)
        return continent_name
    except KeyError:
        pass


def merge_datasets(df: pd.DataFrame) -> pd.DataFrame:
    df["f1"]  = 2 * df.TP / (2 * df.TP + df.FP + df.FN)
    df["precision"] = df.TP / (df.TP + df.FP)
    df["recall"] = df.TP / (df.TP + df.FN)

    df["continent"] = df["country"].apply(country_to_continent)
    df = df[~df.continent.isna()]

    hdi_df = pd.read_csv("data/human_development_index.csv")[["iso3", "hdi_2020"]]
    df = pd.merge(left=df, right=hdi_df, how="left", left_on="gid", right_on="iso3")
    df = df[~df.hdi_2020.isna()].rename(columns={"hdi_2020": "hdi"}).drop(columns="iso3")

    urbpop_df = pd.read_csv("data/urban_population.csv")
    urbpop_df = urbpop_df[urbpop_df["Indicator Code"] == "SP.URB.TOTL.IN.ZS"]
    urbpop_df = urbpop_df[["Country Code", "2020"]].rename(columns={"2020": "urban_population"})
    df = pd.merge(df, urbpop_df, how="left", left_on="gid", right_on="Country Code")
    df = df.drop(columns="Country Code")

    min_size, max_size = 10, 40
    pixel_min = df['pixel_count'].min()
    pixel_max = df['pixel_count'].max()
    df['size'] = ((df['pixel_count'] - pixel_min) / (pixel_max - pixel_min)) * (max_size - min_size) + min_size

    return df

## Results

As shown below, when applying a 1% threshold for built-up areas, a clear association emerges between a country’s development level and its F1 score. Under the relaxed criterion for classifying pixels as built-up, the correspondence between NTL data and built-up surface is systematically higher in more developed countries. As shown in the precision and recall charts below, the proportion of correct predictions among all positive predictions (precision) is roughly similar for Europe and Africa. The key difference arises in recall, where the proportion of correctly identified positives among all actual positives is noticeably lower for Africa, indicating that built-up areas there are more frequently missed in NTL data.

In [3]:
df = pd.read_parquet("results/global_1_percent_thresh")
df = merge_datasets(df)
plot_scatter_bokeh(df)

In [4]:
df = pd.read_parquet("results/global_1_percent_thresh")
df = merge_datasets(df)
plot_scatter_bokeh(df, x="urban_population", xlabel="Percentage of Urban Population")

In [4]:
plot_scatter_bokeh(df, y="precision")

In [5]:
plot_scatter_bokeh(df, y="recall")

However, the results change significantly when applying a 20% threshold, which we introduce to focus on more densely built-up locations. This higher threshold is often recommended for rural or sparsely built-up regions and allows us to test how robust the NTL–built-up correspondence is under a stricter definition of built-up surface. In this case, the relationship between the F1 score and the HDI reverses compared to the 1% threshold. In other words, using a more “strict” criterion for classifying a pixel as built-up increases the agreement between NTL data and built-up areas in Africa, reaching levels comparable to those observed for Europe under the 1% threshold.

The dynamics of precision and recall also change under the 20% threshold (see below). Precision follows the same pattern as the F1 scores: it is clearly lower for Europe, as NTL data tend to identify more built-up areas than indicated by the stricter binary built-up map, leading to an “overshooting” effect. Recall, on the other hand, shows that nearly all built-up areas in Europe - and a substantial proportion in Africa - are successfully captured. Taken together, these patterns suggest that the choice of threshold, and thus how strictly built-up areas are defined, affects European and African countries differently, with African NTL agreeing more closely with more strictly defined, densely built-up locations.

In [6]:
df = pd.read_parquet("results/global_20_percent_thresh")
df = merge_datasets(df)
plot_scatter_bokeh(df)

In [7]:
plot_scatter_bokeh(df, y="precision")

In [8]:
plot_scatter_bokeh(df, y="recall", legend_loc="bottom_right")